In [1]:
# import libraries
!pip install pytesseract
import pytesseract
from PIL import Image
import pandas as pd

In [2]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

# define folders
goget_country_folder = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/'
country = 'Brazil (GOGET O&G)/' # change for country you are working on
screenshots = goget_country_folder + country + 'Production 2025/Screenshots/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# add screenshots from https://app.powerbi.com/view?r=eyJrIjoiZjZhZDliMTYtOWIyZi00OGY5LWJkYzItOTQ1MzFjZGMzMDNkIiwidCI6IjQ0OTlmNGZmLTI0YTYtNGI0Mi1iN2VmLTEyNGFmY2FkYzkxMyJ9
files = [
    "Jan 2025-1.png",

#    "oil production 2.png",
#    "oil production 3.png",
#    "oil production 4.png",
#    "oil production 5.png"
]

In [4]:
# correct accents
accent_corrections = {
    "Agua": "Água",
    "Sabia": "Sabiá",
    "Sabiá Bico-De-Osso": "Sabiá Bico-de-Osso",
    "Sabiá da Mata": "Sabiá da Mata",
    "Acaja": "Acajá",
    "Açaja": "Acajá",
    "Araçás": "Araçás",
    "Araçás Leste": "Araçás Leste",
    "Azulao": "Azulão",
    "Camacari": "Camaçari",
    "Canario": "Canário",
    "Pajeu": "Pajeú",
    "Sao": "São"
}

In [5]:
# extract text from screenshots

all_text = []
for file in files:
    img_path = screenshots + file # prepend the path to the file
    img = Image.open(img_path)
    text = pytesseract.image_to_string(img, lang="eng")  # English OCR still works well
    all_text.append(text)

combined_text = "\n".join(all_text)

In [6]:
combined_text

'Producao petrdleo\n\n(bbI/d)\n\nLeelee ET\ntotal (mil m?/d)\n\nProducao petrdleo\nYe Eat Co) 104)\n\nPercentual\nco2\n\nProducgao\nAgua (m?/d)\n\n \n\nTupi\n\nBuzios\n\nMero\n\nSépia\n\nItapu\n\nSapinhoa\nAtapu\n\nJubarte\nRoncador\nPeregrino\nMarlim Sul\nBerbigao\nMarlim Leste\nLapa\n\nTartaruga Verde\nRio Urucu\nFrade\n\nMarlim\n\nLeste do Urucu\nAlbacora Leste\nBauna\nArgonauta\nMexilhao\nBarracuda\nAtlanta\nPapa-Terra\nCaratinga\nPampo\n\nPilar\n\nMarimba\nGolfinho\nEspadarte\nCarmopolis\nVermelho\nCanto do Amaro\nOstra\n\nPolvo\n\nCarapeba\nGavidao Preto\nAraracanga\nTambuata\nTubarao Martelo\nGavido Real\nMiranga\n\nTié\n\nAzulao\n\nFurado\n\nBonito\n\nPargo\n\nArara Azul\nPitangola\n\nAgua Grande\nXisto SAo Mateus do Sul\nGaviado Vermelho\nTartaruga Verde Sudoeste\nEstreito\n\nAracas\n\nGaviado Caboclo\nGavido Branco\nSalina Cristal\nFazenda Alegre\nPeroa\n\nLorena\n\nMacau\n\nEnchova\n\nAlto do Rodrigues\n\n838,889.10\n652,441.54\n439,875.65\n146,963.66\n157,631.23\n123,505.83

In [7]:
import re

def extract_field_production_final(text, accent_corrections):
    lines = text.splitlines()
    potential_field_names = []
    potential_production_values = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Check if the line strictly matches the pattern of a production value
        if re.fullmatch(r'^[,\d]+\.?\d*$', line):
            try:
                value = float(line.replace(",", ""))
                potential_production_values.append(value)
            except ValueError:
                # If conversion fails, even with strict match, treat as potential field name
                potential_field_names.append(line)
        else:
            # If it doesn't strictly match the value pattern, treat as a potential field name
            potential_field_names.append(line)

    # Filter out likely headers and noise from potential field names
    field_names = []
    for name in potential_field_names:
        # Filter out empty strings and common headers/noise observed in the text
        if name and name.lower() not in ['campo', 'petrdleo (bbl/d)', 'producao', 'producao\npetrdleo (bbl/d)', ' ', '\x0c']:
            field_names.append(name)

    # Apply accent corrections to extracted field names
    corrected_field_names = [accent_corrections.get(name, name) for name in field_names]

    # At this point, we have two lists: corrected_field_names and potential_production_values.
    # The challenge is that they are not in corresponding order from the original text.
    # We need to assume a strict sequential pairing after cleaning.

    # Ensure lists have the same length by truncating to the minimum length
    min_len = min(len(corrected_field_names), len(potential_production_values))
    if len(corrected_field_names) != len(potential_production_values):
        print(f"Warning: Mismatch between number of extracted field names ({len(corrected_field_names)}) and production values ({len(potential_production_values)}). Truncating to {min_len}.")

    # Pair the corrected field names and production values based on their order
    extracted_data = list(zip(corrected_field_names[:min_len], potential_production_values[:min_len]))

    return extracted_data

# Apply the final parsing function to the combined_text from first screenshots
extracted_data_final = extract_field_production_final(combined_text, accent_corrections)

# Create a new DataFrame with the final data
df_final = pd.DataFrame(extracted_data_final, columns=["Campo", "Producao_petroleo_bbl_d"])

In [8]:
# add in additional screenshots and create dataframes for each using the same process
additional_files = [
    "Jan 2025-2.png",
    "Jan 2025-3.png",
    "Jan 2025-4.png",
]

In [9]:
# extract text from additional screenshots

aditional_text = []
for file in additional_files:
    img_path = screenshots + file # prepend the path to the file
    img = Image.open(img_path)
    text = pytesseract.image_to_string(img, lang="eng")  # English OCR still works well
    aditional_text.append(text)

additional_combined_text = "\n".join(aditional_text)

In [10]:
# Apply the final parsing function to the combined_text from additonal screenshots
extracted_data_additional = extract_field_production_final(additional_combined_text, accent_corrections)

# Create a new DataFrame with the additional data
df_additional = pd.DataFrame(extracted_data_additional)

In [11]:
# append df_additional to df_final
df_final = pd.concat([df_final, df_additional], ignore_index=True)

In [12]:
df_final

,Campo,Producao_petroleo_bbl_d,0,1
0,Producao petrdleo,838889.10,NaN,NaN
1,(bbI/d),652441.54,NaN,NaN
2,Leelee ET,439875.65,NaN,NaN
3,total (mil m?/d),146963.66,NaN,NaN
4,Producao petrdleo,157631.23,NaN,NaN
...,...,...,...,...
314,NaN,NaN,Camacari 0.00 3.47 21.82 0.00 0.00,0.00
315,NaN,NaN,Santana 17.74 0.51 20.92 0.24 10.28,0.03
316,NaN,NaN,Joao de Barro 18.74 0.32 20.72 0.00 0.90,0.65
317,NaN,NaN,Pedrinhas 18.57 0.31 20.53 0.00 3.20,0.93


In [ ]:
# save df out to a csv in the screenshots folder
df_final.to_csv(screenshots +'oilscreenshotstotable.csv')

In [ ]:
# do the same thing for gas
gas_files = [
    "gas production 1.png",
    "gas production 2.png",
    "gas production 3.png",
    "gas production 4.png"
]

In [ ]:
# extract text from screenshots

gas_text = []
for file in gas_files:
    img_path = screenshots + file # prepend the path to the file
    img = Image.open(img_path)
    text = pytesseract.image_to_string(img, lang="eng")  # English OCR still works well
    gas_text.append(text)

gas_combined_text = "\n".join(gas_text)

In [ ]:
gas_combined_text

' \n\nCampo Producao gas\nnatural (mil m3/d)\n\nTupi 493,484.53\nBuzios 340,364.63\nMero 247,828.95\nSapinhoa 79,374.15\nRio Urucu 77,785.44\nLeste do Urucu 76,520.22\nSépia 62,374.21\nAtapu 57,016.43\nMexilhao 41,434.86\nJubarte 34,456.61\nRoncador 27,820.63\nSul de Tupi 27,337.20\nSururu 24,262.86\nLapa 19,559.72\nGavidao Preto 16,611.19\nSul de Berbigao 16,534.85\nBerbigao 14,858.96\nItapu 12,887.57\nMarlim Sul 12,113.57\nItapu_Eco 12,036.18\nOeste de Atapu 11,840.13\nMarlim 11,211.27\nAraracanga 10,409.93\nGavido Branco 10,145.28\nAnc_Mero 8,988.61\nSépia Leste 8,562.81\nAzulao 8,134.95\nGaviado Caboclo 7,559.83\nMiranga 7,510.14\nPilar 7,269.90\nTartaruga Verde 7,235.34\nBarracuda 6,434.15\nGavido Real 5,828.72\nFurado 5,694.13\nAlbacora Leste 5,652.29\nArara Azul 5,218.84\nFrade 5,201.34\nPeroa 4,819.25\nAgua Grande 4,803.78\nManati 4,695.96\nGaviado Vermelho 4,087.45\nGavido Tesoura 3,803.21\nSocorro 3,380.39\nLorena 3,114.39\nCaratinga 2,964.82\nArgonauta 2,727.91\nAracas 2,468

In [ ]:
import re

def extract_gas_field_production_revised(text, accent_corrections):
    lines = text.splitlines()
    extracted_data = []
    campo_name = None

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Check if the line contains a potential production value
        value_match = re.search(r'([\d,]+\.?\d*)$', line) # Anchor the regex to the end of the line

        if value_match:
            try:
                value_str = value_match.group(1).replace(",", "")
                value = float(value_str)

                # The part of the line before the number is the field name
                name_part = line[:value_match.start()].strip()

                # If we have a name part from this line, use it. Otherwise, use the accumulated campo_name.
                # This assumes a value is always preceded by its name on the same line, or the previous line(s) built up the name.
                # Given the previous issues, anchoring to the end and taking the preceding text seems more reliable.
                if name_part:
                    campo_name = name_part

                if campo_name:
                    # Apply accent corrections
                    corrected_campo = accent_corrections.get(campo_name, campo_name)
                    extracted_data.append((corrected_campo, value))
                    campo_name = None # Reset after successful pairing
                # If no campo_name and a value is found, it's an unpairable value, ignore for now.

            except ValueError:
                # If conversion fails, treat the whole line as a potential campo name
                if campo_name is None:
                    campo_name = line
                else:
                    campo_name += " " + line # Append to current campo name

        else:
            # If no value found, treat the whole line as a potential campo name or part of one
            if campo_name is None:
                campo_name = line
            else:
                campo_name += " " + line # Append to current campo name

    # A final pass to clean up the extracted data, removing likely headers or noise
    cleaned_data = [(campo, value) for campo, value in extracted_data if "Campo Producao gas" not in campo]

    return cleaned_data

# Apply the revised parsing function
extracted_data_gas_revised = extract_gas_field_production_revised(gas_combined_text, accent_corrections)

# Create a new DataFrame with the revised data
df_gas = pd.DataFrame(extracted_data_gas_revised, columns=["Campo", "Gas production (mil cubic meters/d)"])

# sort df_gas by gas production in descending order
df_gas = df_gas.sort_values(by="Gas production (mil cubic meters/d)", ascending=False)

# Display the resulting DataFrame
display(df_gas)

,Campo,Gas production (mil cubic meters/d)
0,Tupi,493484.53
1,Buzios,340364.63
2,Mero,247828.95
3,Sapinhoa,79374.15
4,Rio Urucu,77785.44
...,...,...
121,Periquito Norte,0.99
122,Sul de Coruripe,0.94
123,Rio Itatinas,0.91
124,Aragari,0.62


In [ ]:
# save df out to a csv in the screenshots folder
df_gas.to_csv(screenshots +'gasscreenshotstotable.csv')

In [ ]:
# sort df_gas by gas production
df_gas = df_gas.sort_values(by="Gas production (mil cubic meters/d)", ascending=False)

df_gas

,Campo,Gas production (mil cubic meters/d)
1,Sapinhoa,1751.84
2,Nordeste,1749.56
6,Esperanca,9.69
7,Estevao,8.78
8,Norte,6.72
9,Rios,5.97
0,Campo Producao gas,3.00
5,Campo Producao gas,3.00
4,Campo Upanema Macau Cupitiba Fazenda Imbé Arap...,3.00
3,Campo Enchova Sanhacgu Boa Esperancga Sabia Bi...,3.00


# Task
Parse the gas production data from the provided screenshots, which has been combined into the `gas_combined_text` variable. Create a pandas DataFrame named `df_gas` with two columns: "Campo" and "Gas production (mil cubic meters/d)". Ensure that all field names and their corresponding gas production values are correctly extracted and paired in the DataFrame. Explain the parsing logic used.

## Analyze the raw gas text patterns

### Subtask:
Carefully examine the `gas_combined_text` output, paying close attention to how field names and production values are structured on lines, how multi-line names are handled, and how entries are separated across the different screenshots.


**Reasoning**:
To analyze the structure of the combined text from the gas production screenshots, I need to print the `gas_combined_text` variable to the console as the first step.



In [ ]:
print(gas_combined_text)

 

Campo Producao gas
natural (mil m3/d)

Tupi 493,484.53
Buzios 340,364.63
Mero 247,828.95
Sapinhoa 79,374.15
Rio Urucu 77,785.44
Leste do Urucu 76,520.22
Sépia 62,374.21
Atapu 57,016.43
Mexilhao 41,434.86
Jubarte 34,456.61
Roncador 27,820.63
Sul de Tupi 27,337.20
Sururu 24,262.86
Lapa 19,559.72
Gavidao Preto 16,611.19
Sul de Berbigao 16,534.85
Berbigao 14,858.96
Itapu 12,887.57
Marlim Sul 12,113.57
Itapu_Eco 12,036.18
Oeste de Atapu 11,840.13
Marlim 11,211.27
Araracanga 10,409.93
Gavido Branco 10,145.28
Anc_Mero 8,988.61
Sépia Leste 8,562.81
Azulao 8,134.95
Gaviado Caboclo 7,559.83
Miranga 7,510.14
Pilar 7,269.90
Tartaruga Verde 7,235.34
Barracuda 6,434.15
Gavido Real 5,828.72
Furado 5,694.13
Albacora Leste 5,652.29
Arara Azul 5,218.84
Frade 5,201.34
Peroa 4,819.25
Agua Grande 4,803.78
Manati 4,695.96
Gaviado Vermelho 4,087.45
Gavido Tesoura 3,803.21
Socorro 3,380.39
Lorena 3,114.39
Caratinga 2,964.82
Argonauta 2,727.91
Aracas 2,468.96
Peregrino 2,411.58
Caburé 2,253.36
Tambuata 2,19

## Identify reliable anchors or patterns

### Subtask:
Identify reliable anchors or patterns in the `gas_combined_text` that can help reliably distinguish field names from production values and correctly pair them.


## Develop a robust gas parsing function

### Subtask:
Create a Python function `extract_gas_field_production_final` that takes the `gas_combined_text` and `accent_corrections` as input and returns a list of (Campo, Gas production) tuples.


**Reasoning**:
Define the `extract_gas_field_production_final` function as per the instructions, iterating through lines, accumulating potential field names, and pairing with production values when found.



In [ ]:
import re

def extract_gas_field_production_final(text, accent_corrections):
    lines = text.splitlines()
    extracted_data = []
    current_campo_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Use a regular expression to search for a pattern that matches the numerical format of production values
        # This regex looks for digits, commas, and an optional decimal part with a period.
        value_match = re.search(r'([\d,]+\.?\d*)', line)

        if value_match:
            try:
                # Extract the matched value string and convert it to float after removing commas
                value_str = value_match.group(1).replace(",", "")
                value = float(value_str)

                # If we have collected campo lines, join them and pair with the value
                if current_campo_lines:
                    campo_name = " ".join(current_campo_lines).strip()
                    if campo_name: # Ensure campo name is not empty after joining
                         # Apply accent corrections if present
                         corrected_campo = accent_corrections.get(campo_name, campo_name)
                         extracted_data.append((corrected_campo, value))
                    current_campo_lines = [] # Reset after pairing

            except ValueError:
                 # If it looked like a value but wasn't, add to potential campo lines
                 current_campo_lines.append(line)

        else:
            # If no value found, add the line to potential campo lines
            current_campo_lines.append(line)

    # After iterating through all lines, we might have remaining campo lines without a value.
    # We will discard these as they don't have corresponding data.

    return extracted_data

## Apply the gas parsing function and create dataframe

### Subtask:
Apply the `extract_gas_field_production_final` function to process the `gas_combined_text` and create the `df_gas` DataFrame.


**Reasoning**:
Apply the parsing function to the combined gas text and create the DataFrame.



In [ ]:
# Apply the final parsing function to the gas_combined_text from additional screenshots
extracted_data_gas = extract_gas_field_production_final(gas_combined_text, accent_corrections)

# Create a new DataFrame with the additional data
df_gas = pd.DataFrame(extracted_data_gas, columns=["Campo", "Gas production (mil cubic meters/d)"])

**Reasoning**:
Display the head of the created DataFrame to verify the data extraction and structure.



In [ ]:
df_gas.head()

,Campo,Gas production (mil cubic meters/d)
0,Campo Producao gas,3.00
1,Sapinhoa,1751.84
2,Nordeste,1749.56
3,Campo Enchova Sanhacgu Boa Esperancga Sabia Bi...,3.00
4,Campo Upanema Macau Cupitiba Fazenda Imbé Arap...,3.00


**Reasoning**:
The DataFrame has been created. However, there are some invalid rows from the header or parsing errors. Filter out these rows and then sort the DataFrame by the gas production values in descending order to complete the subtask and show the result.



In [ ]:
# Filter out rows where 'Campo' likely represents a header or invalid entry
df_gas = df_gas[~df_gas['Campo'].str.contains("Campo Producao gas", na=False)]

# sort df_gas by gas production in descending order
df_gas = df_gas.sort_values(by="Gas production (mil cubic meters/d)", ascending=False)

# Display the filtered and sorted DataFrame
df_gas.head()

,Campo,Gas production (mil cubic meters/d)
1,Sapinhoa,1751.84
2,Nordeste,1749.56
6,Esperanca,9.69
7,Estevao,8.78
8,Norte,6.72


## Validate the `df gas` dataframe

### Subtask:
Thoroughly inspect the `df_gas` DataFrame to ensure all field names and their corresponding gas production values are present and correctly paired.


**Reasoning**:
Display the entire df_gas DataFrame to visually inspect its contents and verify the parsing and pairing.



In [ ]:
display(df_gas)

,Campo,Gas production (mil cubic meters/d)
1,Sapinhoa,1751.84
2,Nordeste,1749.56
6,Esperanca,9.69
7,Estevao,8.78
8,Norte,6.72
9,Rios,5.97
3,Campo Enchova Sanhacgu Boa Esperancga Sabia Bi...,3.00
4,Campo Upanema Macau Cupitiba Fazenda Imbé Arap...,3.00
10,Branco,0.99


**Reasoning**:
The previous display shows that the parsing logic still has issues, including incomplete field names and incorrect pairings. The current parsing logic which relies on joining lines until a number is found is not robust enough for the observed inconsistencies. A different approach is needed.



In [ ]:
import re

def extract_gas_field_production_revised(text, accent_corrections):
    lines = text.splitlines()
    extracted_data = []
    campo_name = None

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Check if the line contains a potential production value
        value_match = re.search(r'([\d,]+\.?\d*)$', line) # Anchor the regex to the end of the line

        if value_match:
            try:
                value_str = value_match.group(1).replace(",", "")
                value = float(value_str)

                # The part of the line before the number is the field name
                name_part = line[:value_match.start()].strip()

                # If we have a name part from this line, use it. Otherwise, use the accumulated campo_name.
                # This assumes a value is always preceded by its name on the same line, or the previous line(s) built up the name.
                # Given the previous issues, anchoring to the end and taking the preceding text seems more reliable.
                if name_part:
                    campo_name = name_part

                if campo_name:
                    # Apply accent corrections
                    corrected_campo = accent_corrections.get(campo_name, campo_name)
                    extracted_data.append((corrected_campo, value))
                    campo_name = None # Reset after successful pairing
                # If no campo_name and a value is found, it's an unpairable value, ignore for now.

            except ValueError:
                # If conversion fails, treat the whole line as a potential campo name
                if campo_name is None:
                    campo_name = line
                else:
                    campo_name += " " + line # Append to current campo name

        else:
            # If no value found, treat the whole line as a potential campo name or part of one
            if campo_name is None:
                campo_name = line
            else:
                campo_name += " " + line # Append to current campo name

    # A final pass to clean up the extracted data, removing likely headers or noise
    cleaned_data = [(campo, value) for campo, value in extracted_data if "Campo Producao gas" not in campo]

    return cleaned_data

# Apply the revised parsing function
extracted_data_gas_revised = extract_gas_field_production_revised(gas_combined_text, accent_corrections)

# Create a new DataFrame with the revised data
df_gas = pd.DataFrame(extracted_data_gas_revised, columns=["Campo", "Gas production (mil cubic meters/d)"])

# sort df_gas by gas production in descending order
df_gas = df_gas.sort_values(by="Gas production (mil cubic meters/d)", ascending=False)

# Display the resulting DataFrame
display(df_gas)

,Campo,Gas production (mil cubic meters/d)
0,Tupi,493484.53
1,Buzios,340364.63
2,Mero,247828.95
3,Sapinhoa,79374.15
4,Rio Urucu,77785.44
...,...,...
121,Periquito Norte,0.99
122,Sul de Coruripe,0.94
123,Rio Itatinas,0.91
124,Aragari,0.62


## Summary:

## Summary of Gas Production Data Parsing

### Data Analysis Key Findings

*   The initial parsing attempt of the `gas_combined_text` variable resulted in an incomplete and incorrectly paired `df_gas` DataFrame, with truncated field names and misassociated production values.
*   A revised parsing logic, implemented in the `extract_gas_field_production_revised` function, significantly improved extraction and pairing by using a regular expression anchored to the end of lines to identify values and extracting the preceding text as field names.
*   The revised parsing successfully generated a `df_gas` DataFrame with 126 rows, indicating a more comprehensive extraction of data compared to the initial attempt.

### Insights or Next Steps

*   The revised parsing function is robust for this specific text structure but may need adjustments for different data formats.
*   The `df_gas` DataFrame is now ready for further analysis or visualization to identify key gas producing fields.
